In [ ]:
from tobii_pytracker.analyze.data_loader import DataLoader
from tobii_pytracker.configs.custom_config import CustomConfig
from tobii_pytracker.datasets.custom_dataset import ImageDataset
from tobii_pytracker.analyze.models import BBoxAttentionAnalyzer

from pathlib import Path
import pandas as pd
from IPython.display import display

config = CustomConfig('../configs/config.yaml')
loader = DataLoader(config, root='../')
print('Processing all samples with bbox generation + gaze mapping')

raw_sets = loader.get_all_data(flatten=False)
gaze_sets = loader.get_all_data(flatten=True)
if not raw_sets:
    raise ValueError('No raw experiment data found.')

non_empty_gaze_sets = {k: v for k, v in gaze_sets.items() if v is not None and not v.empty}
if not non_empty_gaze_sets:
    raise ValueError('No gaze samples found in any run.')

raw_data = pd.concat(raw_sets.values(), keys=raw_sets.keys(), names=['set_name','row_index']).reset_index(level='set_name').reset_index(drop=True)
raw_data['slide_index'] = raw_data.groupby('set_name').cumcount()
gaze_data = pd.concat(non_empty_gaze_sets.values(), ignore_index=True)

raw_data['set_name'] = raw_data['set_name'].astype(str)
gaze_data['set_name'] = gaze_data['set_name'].astype(str)
raw_data['slide_index'] = pd.to_numeric(raw_data['slide_index'], errors='coerce').astype('Int64')
gaze_data['slide_index'] = pd.to_numeric(gaze_data['slide_index'], errors='coerce').astype('Int64')

sets_with_gaze = set(gaze_data['set_name'].dropna().unique())
raw_data = raw_data[raw_data['set_name'].isin(sets_with_gaze)].copy()
gaze_data = gaze_data[gaze_data['set_name'].isin(sets_with_gaze)].copy()
if raw_data.empty:
    raise ValueError('No raw rows remain after filtering runs with gaze.')

gaze_counts = gaze_data.groupby(['set_name', 'slide_index']).size().rename('gaze_count').reset_index()
candidate_rows = raw_data.merge(gaze_counts, on=['set_name', 'slide_index'], how='inner')
candidate_rows = candidate_rows[candidate_rows['gaze_count'] > 0]
if candidate_rows.empty:
    raise ValueError('No slide with gaze points was found.')

print(f'Found {len(candidate_rows)} samples with gaze data')

output_dir = Path('./analysis_outputs/bbox_all_samples')
output_dir.mkdir(parents=True, exist_ok=True)
bbox_analyzer = BBoxAttentionAnalyzer(output_folder=output_dir)

dataset = ImageDataset(config=config, calculate_bboxes=False)

all_comparison_rows = []

for idx, selected_raw in candidate_rows.iterrows():
    set_name = str(selected_raw['set_name'])
    slide_index = int(selected_raw['slide_index'])
    
    print(f'\nProcessing sample {idx + 1}/{len(candidate_rows)}: {set_name}, slide {slide_index}')
    
    selected_gaze = gaze_data[(gaze_data['set_name'] == set_name) & (gaze_data['slide_index'] == slide_index)].copy()
    if selected_gaze.empty:
        print(f'  WARNING: No gaze data found, skipping')
        continue

    screenshot_path = Path(loader.root) / str(selected_raw['screenshot_file'])
    if not screenshot_path.exists():
        screenshot_path = Path(str(selected_raw['screenshot_file']))
    if not screenshot_path.exists():
        print(f'  WARNING: Screenshot not found: {screenshot_path}, skipping')
        continue

    input_image_path = Path(loader.root) / str(selected_raw['input_data'])
    if not input_image_path.exists():
        input_image_path = Path(str(selected_raw['input_data']))
    if not input_image_path.exists():
        print(f'  WARNING: Input image not found: {input_image_path}, skipping')
        continue

    print(f'  Gaze points: {len(selected_gaze)}')

    generated_by_method = {
        'superpixel': dataset._detect_superpixels(str(input_image_path)),
        'grid': dataset._detect_grid(str(input_image_path)),
    }
    try:
        saliency_bboxes = dataset._detect_saliency(str(input_image_path))
        if saliency_bboxes:
            generated_by_method['saliency'] = saliency_bboxes
    except Exception as exc:
        print(f'  Saliency skipped: {exc}')

    for method, bboxes in generated_by_method.items():
        print(f'  {method}: {len(bboxes)} bboxes')

    sample_comparison_rows = []
    saved_plots = {}
    
    for method, method_bboxes in generated_by_method.items():
        method_raw = pd.DataFrame([selected_raw.to_dict()])
        method_raw['set_name'] = method_raw['set_name'].astype(str)
        method_raw['slide_index'] = pd.to_numeric(method_raw['slide_index'], errors='coerce').astype('Int64')
        method_raw.at[0, 'objects_bboxes'] = {'image_bboxes': method_bboxes}

        scores = bbox_analyzer.analyze(raw_data=method_raw, gaze_data=selected_gaze, use_fixations=False)
        if scores.empty:
            print(f'  WARNING: no scores for {method}')
            continue

        evaluation = bbox_analyzer.evaluate(scores)
        if evaluation.empty:
            print(f'  WARNING: no evaluation for {method}')
            continue

        row = evaluation.iloc[0].to_dict()
        row['method'] = method
        row['generated_bbox_count'] = len(method_bboxes)
        row['sample_set_name'] = set_name
        row['sample_slide_index'] = slide_index
        sample_comparison_rows.append(row)
        all_comparison_rows.append(row)

        plot_path = output_dir / f'{set_name}_slide_{slide_index}_{method}.png'
        bbox_analyzer.plot_analysis(
            scored_bboxes=scores,
            gaze_data=selected_gaze,
            screenshot_path=screenshot_path,
            set_name=set_name,
            slide_index=slide_index,
            top_k=25,
            min_hits=1,
            title=f'{method} bboxes - {set_name} slide {slide_index}',
            save_path=plot_path,
        )
        saved_plots[method] = plot_path

    if sample_comparison_rows:
        sample_comparison = pd.DataFrame(sample_comparison_rows)
        print(f'\n  Results for {set_name}, slide {slide_index}:')
        display(sample_comparison[['method', 'generated_bbox_count', 'total_gaze_points', 'bbox_hit_count', 'unique_gaze_hit_count', 'coverage_by_bboxes', 'overlap_factor', 'attended_bboxes', 'attended_bbox_ratio']].sort_values('coverage_by_bboxes', ascending=False))

if all_comparison_rows:
    print(f'Overall Summary across all {len(candidate_rows)} samples:')
    all_comparison = pd.DataFrame(all_comparison_rows)

    summary = all_comparison.groupby('method').agg({
        'generated_bbox_count': 'mean',
        'total_gaze_points': 'sum',
        'bbox_hit_count': 'sum',
        'unique_gaze_hit_count': 'sum',
        'coverage_by_bboxes': 'mean',
        'overlap_factor': 'mean',
        'attended_bboxes': 'mean',
        'attended_bbox_ratio': 'mean',
    }).reset_index()
    
    display(summary.sort_values('coverage_by_bboxes', ascending=False))
    
    print(f'\n✓ Processed {len(all_comparison_rows)} method-sample combinations')
    print(f'✓ Saved plots to: {output_dir}')
else:
    print('WARNING: No valid results were generated')

print('\nDone')